In [3]:
using NNlib
using PyCall
using IterativeSolvers
using LinearMaps
using BenchmarkTools
using TimerOutputs
using StaticArrays
np=pyimport("numpy")

const base_length = [16,16,16]

3-element Vector{Int64}:
 16
 16
 16

In [62]:
function apply_stencil(u; diag = 1)
    i, j, k = size(u)
    result = zeros(Float64, size(u))
    result[2:end-1, 2:end-1, 2:end-1] .= (
        .+ 6 .* u[2:end-1, 2:end-1, 2:end-1]
        .- u[3:end, 2:end-1, 2:end-1] .- u[1:end-2, 2:end-1, 2:end-1]
        .- u[2:end-1, 3:end, 2:end-1] .- u[2:end-1, 1:end-2, 2:end-1]
        .- u[2:end-1, 2:end-1, 3:end] .- u[2:end-1, 2:end-1, 1:end-2]
        
    ) / diag
    return result
end

function subtract_apply_stencil(result, u, rhs; diag)
    n, m, p = size(u)

    @inbounds Threads.@threads for k = 2:p-1
        @inbounds for j=2:m-1
            @simd for i=2:n-1
                result[i, j, k] = rhs[i,j,k] - (
                    6*u[i, j, k]
                    -u[i+1, j, k]-u[i-1, j, k]
                    -u[i, j+1, k]-u[i, j-1, k]
                    -u[i, j, k+1]-u[i, j, k-1]
                ) / diag
            end
        end
    end
end


function compute_res(u, rhs, diag)
    #print("The factor diag in compute res is: ", diag)
    temp = apply_stencil(u; diag = diag) .- rhs
    return sqrt(sum(temp .* temp)) / sqrt(sum(rhs .* rhs))
end


function rb_gauss_seidel_3d(u::Array{Float64, 3}, f::Array{Float64, 3}; diag::Real =1, iterations::Int=3)
    #print("The factor diag in rb_gauss_seidel_3d is: ", diag)
    n, m, p = size(u)
    for _ = 1:iterations
        # Red update: (i + j + k) % 2 == 0
           Threads.@threads for k = 2:p-1
            for j = 2: m - 1 
                 @simd for i = 2:n - 1
                    if (i + j + k) % 2 == 1
                        u[i, j, k] = (1/6) * (
                            u[i+1, j, k] + u[i-1, j, k] +
                            u[i, j+1, k] + u[i, j-1, k] +
                            u[i, j, k+1] + u[i, j, k-1] +
                            diag * f[i, j, k]
                        )

                    end
                end
            end
        end
           # Black update: (i + j + k) % 2 == 1
         Threads.@threads for k=2:p-1
             for j=2:m-1
               @simd for i=2:n - 1
                    if (i + j + k) % 2 == 0
                        u[i, j, k] =  (1/6) * (
                            u[i+1, j, k] + u[i-1, j, k] +
                            u[i, j+1, k] + u[i, j-1, k] +
                            u[i, j, k+1] + u[i, j, k-1] +
                            diag * f[i, j, k]
                        )
                    end
                end
           end
        end
    end
        
end


function restrict(res)
    return res[1:2:end, 1:2:end, 1:2:end]
end

function prolong(coarse)
    i, j, k = size(coarse)
    temp = reshape(coarse, (size(coarse)..., 1, 1))
    temp2 = upsample_trilinear(temp; size=(2*i-1, 2*j-1, 2*k-1))
    return reshape(temp2, (2*i-1, 2*j-1, 2*k-1))
end



function prolong_helper(array, indices_un::NTuple{N, Int},
         res::MVector{N, Int},  
         dim::Int=1) where {N}
    if dim == N+1
        return array[(div.(res, 2) .+ 1)...]
    end

    i = indices_un[dim]
    res[dim] = i

    if isodd(i)
        return prolong_helper(array, indices_un, res, dim + 1)
    else
        res[dim] = i - 1
        val1 = prolong_helper(array, indices_un, res, dim + 1)

        res[dim] = i + 1
        val2 = prolong_helper(array, indices_un, res, dim + 1)

        return (val1 + val2) / 2
    end
end



function prolong_in_place_add(dest, source)
    m, n, p = size(dest)
    @inbounds Threads.@threads for k = 1:p-1
        @inbounds for j = 1:n-1
            @simd for i = 1:m-1
                sv = MVector{3, Int}((0,0,0))
                dest[i,j,k] += prolong_helper(source, (i, j, k), sv)
            end
        end
    end
end

prolong_in_place_add (generic function with 1 method)

In [63]:
function set_boundary_conditions(u)
    i, j, k = size(u)
    for x =1:i
        for y = 1:j
            u[x, y, end]=1
            u[x, y, 1] = 0
        end
    end

    for x =1:i
        for z =1:k
            u[x, end, z] = (z-1) / (k-1)
            u[x, 1, z] = (z-1) / (k-1)
        end 
    end

    for y = 1:j
        for z = 1:k
            u[end, y, z] = (z-1) / (k-1)
            u[1, y, z] = (z-1) / (k-1)
        end
    end
end

function matvec_shape(x_flat, shape, x_grid; factor=2)
    #x_grid = zeros(Float64,shape[1]+2, shape[2]+2, shape[3]+2)
    x_grid[2:end-1, 2:end-1, 2:end-1] = reshape(x_flat, shape)
    Ax_grid = apply_stencil(x_grid) ./ factor
    return vec(Ax_grid[2:end-1, 2:end-1, 2:end-1])
end

matvec_shape (generic function with 1 method)

In [51]:
to = TimerOutput()

────────────────────────────────────────────────────────────────────
                           Time                    Allocations      
                  ───────────────────────   ────────────────────────
Tot / % measured:      316ms /   0.0%           31.9MiB /   0.0%    

Section   ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────

In [71]:

function multigrid_V_cycle(base_length; num_iters=[1 , 1], num_smooth = 2, num_levels = nothing, level_num = nothing, rhs = nothing)

    if (num_levels===nothing || level_num===nothing)
        num_levels=level_num=size(num_iters)[1]
    end

    if(size(num_iters)[1] == 1)
        num_iters = [num_iters[1] for i = 1:num_levels]
    end
    
    length_by_level = [2^n .* base_length .+ 1 for n=1:level_num + 1]

    #Allocating working array
    guess = zeros(Float64, length_by_level[level_num]...)
    u = zeros(Float64, length_by_level[level_num]...)

    #if in uppermost level, set up the problem
    if rhs === nothing       
        #setting boundary conditions       
        set_boundary_conditions(u)
        #print(u)
        #Allocating right hand side
        rhs = -apply_stencil(u)
    end



    #getting matvec function and linear operator
    if level_num == 1
        @timeit to "Inner Conjugate gradient" begin
         x_grid = zeros(Float64, length_by_level[1]...)
         linear_function = x_flat -> matvec_shape(x_flat, Tuple(length_by_level[1] .- 2), x_grid; factor = 2^(num_levels-level_num))
         n = (2*base_length[1]-1)*(2*base_length[2]-1)*(2*base_length[3]-1)
         A = FunctionMap{Float64,false}(linear_function, n, n);
         unravel_c = reshape(rhs[2:end-1, 2:end-1, 2:end-1], :)
         #solving #coarse_grid
         sol = cg(A, unravel_c)
         #print (sol)
         sol_shaped = reshape(sol, Tuple(2 .* base_length .- 1))
         rhs[2:end-1, 2:end-1, 2:end-1] = sol_shaped
        end
        
         return rhs
    end




    #Now doing the multigrid operation
    defect_domain=zeros(size(rhs))
    residuals = zeros(num_iters[1])
    for i = 1:num_iters[1]
        @timeit to "Full cycle" begin
        if level_num==num_levels
            residuum = compute_res(guess, rhs, 2^(num_levels-level_num))
            println("After ", i, " iterations the residual is ", residuum)
            residuals[i]=residuum
        end

        #print("The current level is: ", level_num)
        @timeit to "RB GS" rb_gauss_seidel_3d(guess, rhs; diag = 2^(num_levels-level_num), iterations = num_smooth)
        #print("The guess after the rb_guass_seidel is: ")
        #print(guess)
        #print("The right hand side for the ")
        #print("The guess after a ", 2, " Gauss-Seidel iteration is: ")
        #print(guess)
        #coarsening
        @timeit to "subapp" subtract_apply_stencil(defect_domain, guess, rhs; diag = 2^(num_levels-level_num))
        #print("The defect domain before coarsening is: ")
        #print(defect_domain)
        @timeit to "coarse" coarse = restrict(defect_domain)
        coarse = multigrid_V_cycle(base_length; num_iters=num_iters[2:end] ,num_smooth=num_smooth ,num_levels = num_levels, 
        level_num = level_num-1, rhs = coarse)

        #refining
       
        #refinemend = prolong(coarse)

        #print("The refined coarse grid solution: ")
        #print(refinemend)

        #Adding correction
        #print (refinemend.shape)
        #@timeit to "app guess" guess[2:end-1, 2:end-1, 2:end-1] .+= prolong(coarse)[2:end-1, 2:end-1, 2:end-1]
        @timeit to "app guess" prolong_in_place_add(guess, coarse)

        #print("After adding the coarse grid correction the guess is:")
        #print(guess)
        
        #doing the post smoothing
        #print("Current level: ", level_num)
        @timeit to "RB GS" rb_gauss_seidel_3d(guess, rhs; diag= 2^(num_levels-level_num), iterations= num_smooth)

        #print("After the post smoothing step the guess is:")
        #print(guess)
        end
    end
        
    
    guess .+= u


    return level_num==num_levels ? (guess, residuals) : guess
end

multigrid_V_cycle (generic function with 1 method)

In [74]:
@time guess, residuals = multigrid_V_cycle(base_length; num_iters=[10, 1]);

After 1 iterations the residual is 1.0
After 2 iterations the residual is 0.05213625757457693
After 3 iterations the residual is 0.0016434398572565605
After 4 iterations the residual is 5.724398881126425e-5
After 5 iterations the residual is 2.00291711310089e-6
After 6 iterations the residual is 7.536033202704554e-8
After 7 iterations the residual is 2.774748788541092e-9
After 8 iterations the residual is 1.130461780944158e-10
After 9 iterations the residual is 4.5463574117060045e-12
After 10 iterations the residual is 2.1006263977047603e-13
  0.867588 seconds (2.67 M allocations: 3.079 GiB, 3.63% gc time)


In [76]:
#base_length = [16,16,16]
@time guess, residuals = multigrid_V_cycle(base_length; num_iters=[10, 1, 1]);

After 1 iterations the residual is 1.0
After 2 iterations the residual is 0.05745543683141366
After 3 iterations the residual is 0.0019321056344463493
After 4 iterations the residual is 6.630135782392029e-5
After 5 iterations the residual is 2.3504720185117147e-6
After 6 iterations the residual is 8.53123035923168e-8
After 7 iterations the residual is 3.116955232944619e-9
After 8 iterations the residual is 1.1944858210570177e-10
After 9 iterations the residual is 4.563829668440753e-12
After 10 iterations the residual is 1.9656981869615173e-13
  1.893360 seconds (23.66 M allocations: 5.980 GiB, 9.78% gc time)


In [ ]:
#base_length = [16,16,16]
to=TimerOutput()
@time guess, residuals = multigrid_V_cycle(base_length; num_iters=[10, 1, 1, 1, 1]);

After 1 iterations the residual is 1.0
After 2 iterations the residual is 0.059735493832536034
After 3 iterations the residual is 0.0020535908317960948
After 4 iterations the residual is 7.163604481865335e-5
After 5 iterations the residual is 2.5627727445206335e-6
After 6 iterations the residual is 9.520613201468193e-8
After 7 iterations the residual is 3.53534869673765e-9


In [70]:
to

────────────────────────────────────────────────────────────────────────────────
                                       Time                    Allocations      
                              ───────────────────────   ────────────────────────
      Tot / % measured:            81.3s /  81.8%            198GiB /  92.9%    

Section               ncalls     time    %tot     avg     alloc    %tot      avg
────────────────────────────────────────────────────────────────────────────────
Full cycle                10    66.5s  100.0%   6.65s    184GiB  100.0%  18.4GiB
  app guess               10    19.3s   29.0%   1.93s   40.0GiB   21.7%  4.00GiB
  RB GS                   20    7.78s   11.7%   389ms    710KiB    0.0%  35.5KiB
  Full cycle              10    4.65s    7.0%   465ms   9.40GiB    5.1%  0.94GiB
    app guess             10    1.88s    2.8%   188ms   5.00GiB    2.7%   512MiB
    Full cycle            10    1.50s    2.3%   150ms   3.76GiB    2.0%   385MiB
      Full cycle          1

In [101]:
u = np.zeros([65, 65, 65])
set_boundary_conditions(u)
rhs = -apply_stencil(u)
guess = np.zeros_like(u)


  0.002532 seconds


Array{Float64, 3}

In [19]:
u=np.zeros([5,5,5])
set_boundary_conditions(u)
rhs = -apply_stencil(u)
rhs_flat = reshape(rhs[2:end-1, 2:end-1, 2:end-1], :)

linear_function = x_flat -> matvec_shape(x_flat, (3,3,3); factor = 1)

A_op = FunctionMap{Float64,false}(linear_function, 3*3*3)

x = cg(A_op, rhs_flat)

y=reshape(x, (3,3,3) )
#x = linear_function(rhs_flat)
#guess = np.zeros_like(u)

#rb_gauss_seidel_3d(guess, rhs, diag=1, iterations=3)

#new = restrict(guess)
#refined = prolong(new)

#vec(new)

u[2:end-1, 2:end-1, 2:end-1]=y
u

5×5×5 Array{Float64, 3}:
[:, :, 1] =
 0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0

[:, :, 2] =
 0.25  0.25  0.25  0.25  0.25
 0.25  0.25  0.25  0.25  0.25
 0.25  0.25  0.25  0.25  0.25
 0.25  0.25  0.25  0.25  0.25
 0.25  0.25  0.25  0.25  0.25

[:, :, 3] =
 0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5

[:, :, 4] =
 0.75  0.75  0.75  0.75  0.75
 0.75  0.75  0.75  0.75  0.75
 0.75  0.75  0.75  0.75  0.75
 0.75  0.75  0.75  0.75  0.75
 0.75  0.75  0.75  0.75  0.75

[:, :, 5] =
 1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0

In [21]:
Threads.nthreads()

14

In [45]:
#function prolong_own_helper(array, indices_un, indices_res=[])
    # println(size(indices_res))
#    if length(indices_res)==ndims(array)
#        return array[ (div.(indices_res, 2) .+ 1) ...];
#        
#    elseif indices_un[1]%2==1
#        return prolong_own_helper(array, 
#               indices_un[2: end], [indices_res...,
#                indices_un[1]])
#    else
#        return (
#                prolong_own_helper(array, 
#                indices_un[2: end], 
#                [indices_res..., indices_un[1]+1]) 
#              + prolong_own_helper(array, indices_un[2: end], 
#                [indices_res...,
#                 indices_un[1]-1])
#                ) / 2
#    end
#end

function prolong_helper(array, indices_un::NTuple{N, Int},
         res::MVector{N, Int},  
         dim::Int=1) where {N}
    if dim == N+1
        return array[(div.(res, 2) .+ 1)...]
    end

    i = indices_un[dim]
    res[dim] = i

    if isodd(i)
        return prolong_helper(array, indices_un, res, dim + 1)
    else
        res[dim] = i - 1
        val1 = prolong_helper(array, indices_un, res, dim + 1)

        res[dim] = i + 1
        val2 = prolong_helper(array, indices_un, res, dim + 1)

        return (val1 + val2) / 2
    end
end



function prolong_own(dest, source)
    m, n, p = size(dest)
    for k = 1:p-1
        for j = 1:n-1
            for i = 1:m-1
                sv = MVector{3, Int}((0,0,0))
                dest[i,j,k] = prolong_helper(source, (i, j, k), sv)
            end
        end
    end
end

prolong_own (generic function with 1 method)

In [46]:
u=np.zeros([7,7,7])
set_boundary_conditions(u)
guess = zeros(Float64, size(u)...)
new_own = zeros(Float64, size(u)...)
rhs = -apply_stencil(u)
rb_gauss_seidel_3d(guess, rhs)
coarse=restrict(guess)
new = prolong(coarse)
prolong_own(new_own, coarse)
new_own

7×7×7 Array{Float64, 3}:
[:, :, 1] =
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0

[:, :, 2] =
 0.0  0.0        0.0        0.0        0.0        0.0        0.0
 0.0  0.0117563  0.0235125  0.0235125  0.0235125  0.0117563  0.0
 0.0  0.0235125  0.047025   0.047025   0.047025   0.0235125  0.0
 0.0  0.0235125  0.047025   0.047025   0.047025   0.0235125  0.0
 0.0  0.0235125  0.047025   0.047025   0.047025   0.0235125  0.0
 0.0  0.0117563  0.0235125  0.0235125  0.0235125  0.0117563  0.0
 0.0  0.0        0.0        0.0        0.0        0.0        0.0

[:, :, 3] =
 0.0  0.0        0.0        0.0        0.0        0.0        0.0
 0.0  0.0235125  0.047025   0.047025   0.047025   0.0235125  0.0
 0.0  0.047025   0.0940501  0.0940501  0.0940501  0.047025   0.0
 0.0  0.047025   0.0940501  0.0940501  0.0

In [84]:
new

7×7×7 Array{Float64, 3}:
[:, :, 1] =
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0

[:, :, 2] =
 0.0  0.0        0.0        0.0        0.0        0.0        0.0
 0.0  0.0117563  0.0235125  0.0235125  0.0235125  0.0117563  0.0
 0.0  0.0235125  0.047025   0.047025   0.047025   0.0235125  0.0
 0.0  0.0235125  0.047025   0.047025   0.047025   0.0235125  0.0
 0.0  0.0235125  0.047025   0.047025   0.047025   0.0235125  0.0
 0.0  0.0117563  0.0235125  0.0235125  0.0235125  0.0117563  0.0
 0.0  0.0        0.0        0.0        0.0        0.0        0.0

[:, :, 3] =
 0.0  0.0        0.0        0.0        0.0        0.0        0.0
 0.0  0.0235125  0.047025   0.047025   0.047025   0.0235125  0.0
 0.0  0.047025   0.0940501  0.0940501  0.0940501  0.047025   0.0
 0.0  0.047025   0.0940501  0.0940501  0.0

In [31]:
sv = SVector{3, Int}(ntuple(i->0, 3))

3-element SVector{3, Int64} with indices SOneTo(3):
 0
 0
 0

In [30]:
a = SVector{3, Int}(0)

LoadError: DimensionMismatch: No precise constructor for SVector{3, Int64} found. Length of input was 1.